# Ark+ 3D — run05  (Models Genesis augmentation + fixed LR + no consistency explosion)
**GPU:** RTX 4060 | **Env:** ark3d | **Repo:** [ArnabmoyPaul/ark_3d_only](https://github.com/ArnabmoyPaul/ark_3d_only)

### What changed from run04
| # | Was broken | Fixed |
|---|---|---|
| 1 | LR stayed at ~1.1e-4 (timm scheduler bug) | Manual cosine warmup — LR actually reaches 1e-3 |
| 2 | Consistency loss exploded to 3×10¹³ at epoch 40 | Removed entirely — pure task loss |
| 3 | Student aug: only flips + noise | **Models Genesis augmentations added** — Bézier non-linear, local pixel shuffling, inner/outer cutout |
| 4 | Output: 7500-line log full of FutureWarnings | One clean line per epoch |

### SOTA targets (goal: ≥ 0.80 on all)
| Dataset | Task | SOTA |
|---------|------|------|
| OrganMNIST3D | 11-class mAUC | 0.997 |
| NoduleMNIST3D | binary AUC | 0.863 |
| AdrenalMNIST3D | binary AUC | 0.874 |
| FractureMNIST3D | 3-class mAUC | 0.714 |
| VesselMNIST3D | binary AUC | 0.914 |
| SynapseMNIST3D | binary AUC | 0.843 |

---
## Cell 1 — Verify GPU

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB')
    print('✓ Ready' if vram >= 6 else '⚠️  < 6 GB — use --batch_size 32')
else:
    print('⚠️  No GPU')

---
## Cell 2 — Clone / Pull Repo

In [ ]:
import os, sys, subprocess

WORK_DIR = r'C:\Users\Arnabmoy\Desktop\ark_3d_only'   # ← change if needed
REPO_URL = 'https://github.com/ArnabmoyPaul/ark_3d_only.git'

os.makedirs(os.path.dirname(WORK_DIR), exist_ok=True)

if not os.path.exists(os.path.join(WORK_DIR, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, WORK_DIR], check=True)
    print('✓ Cloned')
else:
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=True)
    print('✓ Pulled latest')

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print('Working dir:', os.getcwd())
print('Files:', sorted(f for f in os.listdir('.') if f.endswith('.py')))

---
## Cell 3 — Install Dependencies

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
                   capture_output=True, text=True)
print('✓ requirements installed' if r.returncode == 0 else f'⚠️  {r.stderr[-300:]}')

for pkg in ['torch', 'timm', 'medmnist', 'einops', 'sklearn', 'matplotlib', 'yaml', 'scipy']:
    try:
        v = __import__(pkg).__version__
        print(f'  ✓ {pkg:<15s} {v}')
    except Exception as e:
        print(f'  ✗ {pkg:<15s} {e}')

---
## Cell 4 — Quick Sanity Test (CPU, 2 epochs)
Run this first. Takes ~2 minutes. Confirms the whole pipeline works before committing to 300 epochs.

In [ ]:
import subprocess, sys

print('Sanity test: NoduleMNIST3D + SynapseMNIST3D, 2 epochs, CPU')
print('─' * 55)

cmd = [
    sys.executable, 'main_3d.py',
    '--datasets',        'NoduleMNIST3D', 'SynapseMNIST3D',
    '--pretrain_epochs', '2',
    '--batch_size',      '16',
    '--lr',              '1e-3',
    '--warmup_epochs',   '0',
    '--test_epoch',      '1',
    '--exp_name',        'sanity',
    '--device',          'cpu',
    '--workers',         '0',
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, encoding='utf-8', errors='replace')
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('─' * 55)
print('✓ Pipeline OK — run Cell 5' if proc.returncode == 0 else f'✗ Failed (code {proc.returncode})')

---
## Cell 5 — Run Training (run05)

**What to expect in the output:**
```
─────────────────────────────────────────────────────────────────
 Epoch   ValLoss        LR  OrganMNI  NoduleMN  AdrenalM  ...
─────────────────────────────────────────────────────────────────
     1    1.0842  1.00e-04         ?         ?         ?   ...
     2    0.9731  2.00e-04         ?         ?         ?   ...
    10    0.7214  9.82e-04         ?         ?         ?   ...

  Epoch 10 test results:
  Dataset                AUC/ACC    SOTA      Gap
  NoduleMNIST3D           0.7234   0.863  -0.1396  ✗
  ...
  Mean AUC: 0.7146
─────────────────────────────────────────────────────────────────
    11    0.7011  9.64e-04         ?  ...
```

**Expected time:** ~3–4 hours on RTX 4060 for 300 epochs.

In [ ]:
import subprocess, sys, os
from datetime import datetime

# ── Settings ─────────────────────────────────────────────────────────────────
EXP_NAME   = 'run05'
EPOCHS     = 300
BATCH_SIZE = 64    # reduce to 32 if CUDA OOM
LR         = '1e-3'
WARMUP     = 10
TEST_EVERY = 10    # print AUC table every N epochs
DEVICE     = 'cuda'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('Logs', exist_ok=True)
logfile = f'Logs/{EXP_NAME}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'

cmd = [
    sys.executable, 'main_3d.py',
    '--pretrain_epochs', str(EPOCHS),
    '--batch_size',      str(BATCH_SIZE),
    '--lr',              LR,
    '--warmup_epochs',   str(WARMUP),
    '--test_epoch',      str(TEST_EVERY),
    '--exp_name',        EXP_NAME,
    '--device',          DEVICE,
]

print(f'Starting: {" ".join(cmd[2:])}')
print(f'Log:      {logfile}')
print()

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, encoding='utf-8', errors='replace')
with open(logfile, 'w', encoding='utf-8') as lf:
    for line in proc.stdout:
        print(line, end='', flush=True)
        lf.write(line); lf.flush()
proc.wait()
print()
print('✓ Done' if proc.returncode == 0 else f'✗ Exited {proc.returncode}')

---
## Cell 6 — Resume (if interrupted)

In [ ]:
import subprocess, sys, os
from datetime import datetime

EXP_NAME   = 'run05'   # must match Cell 5
EPOCHS     = 300
BATCH_SIZE = 64
LR         = '1e-3'
WARMUP     = 10
TEST_EVERY = 10
DEVICE     = 'cuda'

os.makedirs('Logs', exist_ok=True)
logfile = f'Logs/{EXP_NAME}_resume_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'

cmd = [
    sys.executable, 'main_3d.py',
    '--pretrain_epochs', str(EPOCHS),
    '--batch_size',      str(BATCH_SIZE),
    '--lr',              LR,
    '--warmup_epochs',   str(WARMUP),
    '--test_epoch',      str(TEST_EVERY),
    '--exp_name',        EXP_NAME,
    '--device',          DEVICE,
    '--resume',
]

print(f'Resuming {EXP_NAME}...')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, encoding='utf-8', errors='replace')
with open(logfile, 'w', encoding='utf-8') as lf:
    for line in proc.stdout:
        print(line, end='', flush=True)
        lf.write(line); lf.flush()
proc.wait()

---
## Cell 7 — Results Table
Reads the CSV written by engine_3d.py and prints the full AUC history.

In [ ]:
import os, glob

EXP_NAME = 'run05'
SOTA = {
    'OrganMNIST3D':   0.997,
    'NoduleMNIST3D':  0.863,
    'AdrenalMNIST3D': 0.874,
    'FractureMNIST3D':0.714,
    'VesselMNIST3D':  0.914,
    'SynapseMNIST3D': 0.843,
}

csv_path = f'Outputs/{EXP_NAME}/results_{EXP_NAME}.csv'
if not os.path.exists(csv_path):
    print(f'No results yet at {csv_path}')
    print('Run at least 10 epochs first (first test_epoch).')
else:
    with open(csv_path) as f:
        lines = [l.strip() for l in f if l.strip()]

    header  = lines[0].split(',')
    ds_cols = header[1:-1]   # skip epoch and avg_val_loss
    rows    = [l.split(',') for l in lines[1:]]

    print(f'{'Epoch':>6}  ' + '  '.join(f'{d[:10]:>10}' for d in ds_cols) + f'  {'MeanAUC':>8}')
    print('─' * (6 + 12*len(ds_cols) + 12))

    for row in rows:
        ep    = row[0]
        aucs  = [float(x) for x in row[1:-1]]
        mean  = sum(aucs) / len(aucs)
        vals  = '  '.join(f'{v:>10.4f}' for v in aucs)
        print(f'{ep:>6}  {vals}  {mean:>8.4f}')

    print()
    print('Best per dataset:')
    best = [max(float(r[i+1]) for r in rows) for i in range(len(ds_cols))]
    print(f'{'Dataset':<22}  {'Best':>6}  {'SOTA':>6}  {'Gap':>7}')
    print('─' * 45)
    for ds, b in zip(ds_cols, best):
        st  = SOTA.get(ds, 0)
        gap = b - st
        flag = '✓' if gap >= -0.03 else ('~' if gap >= -0.10 else '✗')
        print(f'{ds:<22}  {b:>6.4f}  {st:>6.3f}  {gap:>+7.4f}  {flag}')

---
## Cell 8 — Generate Plots

In [ ]:
import subprocess, sys

EXP_NAME = 'run05'

subprocess.run([sys.executable, 'plot_results.py',
                '--exp_name', EXP_NAME, '--out_dir', 'Plots'])
print('Plots saved to Plots/ — click any PNG in VS Code Explorer to preview.')